# Lesson 1: Advanced RAG Pipeline

In [1]:
import utils

import os
import openai
openai.api_key = utils.get_openai_api_key()

✅ In Answer Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Answer Relevance, input response will be set to __record__.main_output or `Select.RecordOutput` .
✅ In Context Relevance, input prompt will be set to __record__.main_input or `Select.RecordInput` .
✅ In Context Relevance, input response will be set to __record__.app.query.rets.source_nodes[:].node.text .
✅ In Groundedness, input source will be set to __record__.app.query.rets.source_nodes[:].node.text .
✅ In Groundedness, input statement will be set to __record__.main_output or `Select.RecordOutput` .


In [2]:
from llama_index import SimpleDirectoryReader

documents = SimpleDirectoryReader(
    input_files=["./eBook-How-to-Build-a-Career-in-AI.pdf"]
).load_data()

In [3]:
print(type(documents), "\n")
print(len(documents), "\n")
print(type(documents[0]))
print(documents[0])

<class 'list'> 

41 

<class 'llama_index.schema.Document'>
Doc ID: c1b30577-1d4c-469b-a46b-6553acf24fda
Text: PAGE 1Founder, DeepLearning.AICollected Insights from Andrew Ng
How to  Build Your Career in AIA Simple Guide


## Basic RAG pipeline

In [4]:
from llama_index import Document

document = Document(text="\n\n".join([doc.text for doc in documents]))

In [5]:
from llama_index import VectorStoreIndex
from llama_index import ServiceContext
from llama_index.llms import OpenAI

llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)
service_context = ServiceContext.from_defaults(
    llm=llm, embed_model="local:BAAI/bge-small-en-v1.5"
)
index = VectorStoreIndex.from_documents([document],
                                        service_context=service_context)

In [6]:
query_engine = index.as_query_engine()

In [7]:
response = query_engine.query(
    "What are steps to take when finding projects to build your experience?"
)
print(str(response))

Steps for finding projects to build your experience:

1. **Generate project ideas**
   - Join an existing project.
   - Keep reading, taking courses, and talking with experts.
   - Look for machine-learning applications in an area you know.
   - Start a side project or “side hustle,” provided it doesn’t conflict with your employer.

2. **Evaluate each idea**
   - Will it help you grow technically without being unrealistically difficult?
   - Will you have good teammates, mentors, or people to discuss it with?
   - Could it become a stepping stone to larger or more complex projects?

3. **Start at an appropriate scale**
   - Don’t worry about beginning with a small project; small successes build skills and credibility.
   - Aim to develop a portfolio that progresses from simple to increasingly complex work.

4. **Choose how much planning is needed**
   - Use **“ready, aim, fire”** when execution is costly and research can reduce risk.
   - Use **“ready, fire, aim”** when building a low-

## Evaluation setup using TruLens

In [8]:
eval_questions = []
with open('eval_questions.txt', 'r') as file:
    for line in file:
        # Remove newline character and convert to integer
        item = line.strip()
        print(item)
        eval_questions.append(item)

What are the keys to building a career in AI?
How can teamwork contribute to success in AI?
What is the importance of networking in AI?
What are some good habits to develop for a successful career?
How can altruism be beneficial in building a career?
What is imposter syndrome and how does it relate to AI?
Who are some accomplished individuals who have experienced imposter syndrome?
What is the first step to becoming good at AI?
What are some common challenges in AI?
Is it normal to find parts of AI challenging?


In [9]:
# You can try your own question:
new_question = "What is the right AI job for me?"
eval_questions.append(new_question)

In [10]:
print(eval_questions)

['What are the keys to building a career in AI?', 'How can teamwork contribute to success in AI?', 'What is the importance of networking in AI?', 'What are some good habits to develop for a successful career?', 'How can altruism be beneficial in building a career?', 'What is imposter syndrome and how does it relate to AI?', 'Who are some accomplished individuals who have experienced imposter syndrome?', 'What is the first step to becoming good at AI?', 'What are some common challenges in AI?', 'Is it normal to find parts of AI challenging?', 'What is the right AI job for me?']


In [11]:
from trulens_eval import Tru
tru = Tru()

tru.reset_database()

🦑 Tru initialized with db url sqlite:///default.sqlite .
🛑 Secret keys may be written to the database. See the `database_redact_keys` option of `Tru` to prevent this.


For the classroom, we've written some of the code in helper functions inside a utils.py file.  
- You can view the utils.py file in the file directory by clicking on the "Jupyter" logo at the top of the notebook.
- In later lessons, you'll get to work directly with the code that's currently wrapped inside these helper functions, to give you more options to customize your RAG pipeline.

In [12]:
from utils import get_prebuilt_trulens_recorder

tru_recorder = get_prebuilt_trulens_recorder(query_engine,
                                             app_id="Direct Query Engine")

In [13]:
with tru_recorder as recording:
    for question in eval_questions:
        response = query_engine.query(question)

In [14]:
records, feedback = tru.get_records_and_feedback(app_ids=[])

In [15]:
records.head()

,app_id,app_json,type,record_id,input,output,tags,record_json,cost_json,perf_json,ts,Answer Relevance,Context Relevance,Groundedness,Answer Relevance_calls,Context Relevance_calls,Groundedness_calls,latency,total_tokens,total_cost
0,Direct Query Engine,"{""app_id"": ""Direct Query Engine"", ""tags"": ""-"",...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_67c1081245fc168a1c654c110c15b19a,"""What are the keys to building a career in AI?""","""The key steps to building a career in AI are:...",-,"{""record_id"": ""record_hash_67c1081245fc168a1c6...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:21:24.625618"", ""...",2026-11-04T22:21:29.060054,1.0,0.90,0.755000,[{'args': {'prompt': 'What are the keys to bui...,[{'args': {'prompt': 'What are the keys to bui...,"[{'args': {'source': 'PAGE 1Founder, DeepLearn...",4,2377,0.003747
1,Direct Query Engine,"{""app_id"": ""Direct Query Engine"", ""tags"": ""-"",...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_ec834e08582dcc1416d4108d6687ed2f,"""How can teamwork contribute to success in AI?""","""Teamwork contributes to success in AI by enab...",-,"{""record_id"": ""record_hash_ec834e08582dcc1416d...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:21:29.185977"", ""...",2026-11-04T22:21:30.945620,1.0,0.40,0.966667,[{'args': {'prompt': 'How can teamwork contrib...,[{'args': {'prompt': 'How can teamwork contrib...,[{'args': {'source': 'Hopefully the previous c...,1,1688,0.002565
2,Direct Query Engine,"{""app_id"": ""Direct Query Engine"", ""tags"": ""-"",...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_4e819943e2e08a4907dbfbdac4938e2e,"""What is the importance of networking in AI?""","""Networking is important in AI because a stron...",-,"{""record_id"": ""record_hash_4e819943e2e08a4907d...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:21:31.070177"", ""...",2026-11-04T22:21:32.667523,0.9,0.40,0.900000,[{'args': {'prompt': 'What is the importance o...,[{'args': {'prompt': 'What is the importance o...,[{'args': {'source': 'Hopefully the previous c...,1,1669,0.002527
3,Direct Query Engine,"{""app_id"": ""Direct Query Engine"", ""tags"": ""-"",...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_42e16a147f0730f22ea37aff6c16daf0,"""What are some good habits to develop for a su...","""Good habits include:\n\n- Eating healthily an...",-,"{""record_id"": ""record_hash_42e16a147f0730f22ea...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:21:32.788955"", ""...",2026-11-04T22:21:35.732897,1.0,0.90,0.900000,[{'args': {'prompt': 'What are some good habit...,[{'args': {'prompt': 'What are some good habit...,[{'args': {'source': 'Hopefully the previous c...,2,1655,0.002511
4,Direct Query Engine,"{""app_id"": ""Direct Query Engine"", ""tags"": ""-"",...",RetrieverQueryEngine(llama_index.query_engine....,record_hash_78222e5548184e8956e547b73d6cfbd9,"""How can altruism be beneficial in building a ...","""Altruism can improve a career because helping...",-,"{""record_id"": ""record_hash_78222e5548184e8956e...","{""n_requests"": 1, ""n_successful_requests"": 1, ...","{""start_time"": ""2026-11-04T22:21:35.887341"", ""...",2026-11-04T22:21:37.847975,0.9,0.35,NaN,[{'args': {'prompt': 'How can altruism be bene...,[{'args': {'prompt': 'How can altruism be bene...,NaN,1,1689,0.002580


In [16]:
tru.get_leaderboard(app_ids=[])

,Groundedness,Answer Relevance,Context Relevance,latency,total_cost
app_id,,,,,
Direct Query Engine,0.880417,0.95,0.59,1.727273,0.003118


In [17]:
# launches on http://localhost:8501/
tru.run_dashboard()

Starting dashboard ...
Config file already exists. Skipping writing process.
Credentials file already exists. Skipping writing process.


Accordion(children=(VBox(children=(VBox(children=(Label(value='STDOUT'), Output())), VBox(children=(Label(valu…

Dashboard started at https://s172-29-35-192p38560.lab-aws-production.deeplearning.ai/ .


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

## Advanced RAG pipeline

### 1. Sentence Window retrieval

In [18]:
from llama_index.llms import OpenAI

llm = OpenAI(model="gpt-3.5-turbo", temperature=0.1)

In [19]:
from utils import build_sentence_window_index

sentence_index = build_sentence_window_index(
    document,
    llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="sentence_index"
)

In [20]:
from utils import get_sentence_window_query_engine

sentence_window_engine = get_sentence_window_query_engine(sentence_index)

In [21]:
window_response = sentence_window_engine.query(
    "how do I get started on a personal project in AI?"
)
print(str(window_response))

To get started on a personal AI project:

1. **Choose a responsible, ethical, and beneficial problem** to work on.
2. **Identify and scope the project** clearly before building it.
3. **Select a project that supports your career goals** and helps you develop relevant skills.
4. **Start at an appropriate level**, even if you don’t yet have deep expertise.
5. **Aim for projects that grow over time** in scope, complexity, and impact, creating a progression you can eventually demonstrate in a portfolio.


In [22]:
tru.reset_database()

tru_recorder_sentence_window = get_prebuilt_trulens_recorder(
    sentence_window_engine,
    app_id = "Sentence Window Query Engine"
)

In [23]:
for question in eval_questions:
    with tru_recorder_sentence_window as recording:
        response = sentence_window_engine.query(question)
        print(question)
        print(str(response))

What are the keys to building a career in AI?
The keys to building a career in AI are:

1. **Learn foundational technical skills**
2. **Work on meaningful projects**
3. **Find the right AI job**
4. **Build a supportive community** of friends and allies who can help—and whom you help in return.
How can teamwork contribute to success in AI?
Teamwork can contribute to success in AI by:

- **Combining diverse expertise:** Teammates can offer different technical insights and help identify flaws in a project’s reasoning.
- **Improving decision-making:** Collaboration can clarify when to adopt a new technical architecture or collect more data.
- **Helping projects grow:** Strong communication about a project’s value can bring colleagues, mentors, and managers onboard.
- **Developing leadership:** Guiding a project through technical contributions builds leadership skills, even without a formal management role.
- **Influencing work habits and values:** Colleagues who work hard, keep learning, a

In [24]:
tru.get_leaderboard(app_ids=[])

,Groundedness,Answer Relevance,Context Relevance,latency,total_cost
app_id,,,,,
Sentence Window Query Engine,0.98,0.95,0.583333,2.545455,0.000875


In [25]:
# launches on http://localhost:8501/
tru.run_dashboard()

Starting dashboard ...
Config file already exists. Skipping writing process.
Credentials file already exists. Skipping writing process.
Dashboard already running at path: https://s172-29-35-192p38560.lab-aws-production.deeplearning.ai/


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>

### 2. Auto-merging retrieval

In [34]:
from utils import build_automerging_index

automerging_index = build_automerging_index(
    documents,
    llm,
    embed_model="local:BAAI/bge-small-en-v1.5",
    save_dir="merging_index"
)

In [35]:
from utils import get_automerging_query_engine

automerging_query_engine = get_automerging_query_engine(
    automerging_index,
)

In [36]:
auto_merging_response = automerging_query_engine.query(
    "How do I build a portfolio of AI projects?"
)
print(str(auto_merging_response))

> Merging 1 nodes into parent node.
> Parent node id: afccf188-4366-410a-b138-c2fc136f2da1.
> Parent node text: PAGE 21Building a Portfolio of 
Projects that Shows 
Skill Progression CHAPTER 6
PROJECTS

> Merging 1 nodes into parent node.
> Parent node id: 201fe554-dacc-42d4-b5c1-18226c9281e6.
> Parent node text: PAGE 21Building a Portfolio of 
Projects that Shows 
Skill Progression CHAPTER 6
PROJECTS

Build a portfolio by:

- Choosing meaningful AI ideas and projects, even in industries where you aren’t yet an expert.
- Applying machine learning to a variety of fields, such as healthcare, manufacturing, climate change, agriculture, ecommerce, or advertising.
- Showing clear progress over time—from simple projects to increasingly complex undertakings.
- Documenting and communicating your thinking so others understand the value of your work and can trust you with larger projects.


In [37]:
tru.reset_database()

tru_recorder_automerging = get_prebuilt_trulens_recorder(automerging_query_engine,
                                                         app_id="Automerging Query Engine")

In [38]:
for question in eval_questions:
    with tru_recorder_automerging as recording:
        response = automerging_query_engine.query(question)
        print(question)
        print(response)

> Merging 2 nodes into parent node.
> Parent node id: b0808ad3-0894-4043-ab76-9cc0d68b3b48.
> Parent node text: PAGE 3Table of 
ContentsIntroduction: Coding AI is the New Literacy.
Chapter 1: Three Steps to Ca...

> Merging 1 nodes into parent node.
> Parent node id: 5468c736-fa14-45cd-b879-1e6834f58957.
> Parent node text: PAGE 3Table of 
ContentsIntroduction: Coding AI is the New Literacy.
Chapter 1: Three Steps to Ca...

What are the keys to building a career in AI?
The keys to building a career in AI are:

- Learning foundational technical skills  
- Working on projects  
- Finding a job  
- Being part of a supportive community  
- Developing the ability to collaborate, influence, and learn from others in teams
How can teamwork contribute to success in AI?
Teamwork contributes to success in AI by helping people handle large projects more effectively than they could alone. Collaborating with others, influencing and accepting influence, and using strong interpersonal and communicatio

What is the right AI job for me?
The right AI job is likely one that lets you transition in either your role or your industry—but not both at once. If you’re seeking your first AI position, switching one dimension at a time may be easier: for example, move into an AI-related role within your current industry, or keep your role while moving to a tech company.


In [39]:
tru.get_leaderboard(app_ids=[])

,Groundedness,Answer Relevance,Context Relevance,latency,total_cost
app_id,,,,,
Automerging Query Engine,0.9,0.855556,0.65,3.090909,0.000925


In [40]:
# launches on http://localhost:8501/
tru.run_dashboard()

Starting dashboard ...
Config file already exists. Skipping writing process.
Credentials file already exists. Skipping writing process.
Dashboard already running at path: https://s172-29-35-192p38560.lab-aws-production.deeplearning.ai/


<Popen: returncode: None args: ['streamlit', 'run', '--server.headless=True'...>